# Praktikum Sains Data — Analisis Regresi (Google Colab)
**Tujuan:** mengembangkan model regresi, memeriksa asumsi, melakukan inferensi, dan mengevaluasi prediksi pada data uji. Semua data utama adalah **simulasi**, bukan hasil penelitian nyata.

**Cara pakai:** unggah berkas `.ipynb` ke https://colab.research.google.com/ dan jalankan **Runtime → Run all**. Seed ditetapkan agar hasil dapat direproduksi.

## 1. Library dan data simulasi pendidikan
Model pembangkitan: nilai = 35 + 3.5×jam_belajar + 0.32×kehadiran + 4×akses_internet + galat. Koefisien adalah parameter simulasi, bukan efek kausal.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_breusch_godfrey, linear_reset
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV
rng=np.random.default_rng(42)
n=300
jam=rng.uniform(1,12,n)
hadir=rng.uniform(60,100,n)
internet=rng.binomial(1,0.65,n)
noise=rng.normal(0,6,n)
nilai=35+3.5*jam+0.32*hadir+4*internet+noise
df=pd.DataFrame({'jam_belajar':jam,'kehadiran':hadir,'akses_internet':internet,'nilai':nilai})
print(df.head().round(2))
print('Missing:\n',df.isna().sum())
print(df.describe().round(2))

## 2. Exploratory Data Analysis (EDA)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].scatter(df.jam_belajar,df.nilai,alpha=.6)
axes[0].set(xlabel='Jam belajar',ylabel='Nilai',title='Hubungan bivariat')
axes[1].hist(df.nilai,bins=20)
axes[1].set(xlabel='Nilai',ylabel='Frekuensi',title='Distribusi nilai')
plt.tight_layout();plt.show()
print(df.corr(numeric_only=True).round(2))

## 3. Estimasi OLS dan interpretasi
Koefisien parsial mengukur perubahan nilai rerata terkait kenaikan satu unit prediktor dengan prediktor lain tetap. Perhatikan interval kepercayaan dan satuan.

In [ ]:
model=smf.ols('nilai ~ jam_belajar + kehadiran + akses_internet',data=df).fit()
print(model.summary())
print('Interval kepercayaan 95%:\n',model.conf_int().round(3))
print('Koefisien HC3 (standard error tahan heteroskedastisitas):')
print(model.get_robustcov_results(cov_type='HC3').summary())

## 4. Diagnostik model
Interpretasi uji: p kecil dapat menjadi indikasi penolakan hipotesis nol; p besar **tidak membuktikan** asumsi terpenuhi. Durbin–Watson/Breusch–Godfrey relevan terutama jika observasi memiliki urutan bermakna.

In [ ]:
fitted=model.fittedvalues
resid=model.resid
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].scatter(fitted,resid,alpha=.6)
axes[0].axhline(0,color='black',linestyle='--')
axes[0].set(xlabel='Fitted',ylabel='Residual',title='Residual vs fitted')
sm.qqplot(resid,line='45',ax=axes[1]);axes[1].set_title('Q–Q plot')
plt.tight_layout();plt.show()
bp=het_breuschpagan(resid,model.model.exog)
print('Breusch–Pagan LM p-value:',round(bp[1],4))
print('Shapiro–Wilk p-value:',round(stats.shapiro(resid).pvalue,4))
print('Durbin–Watson:',round(sm.stats.stattools.durbin_watson(resid),3))
print('Breusch–Godfrey p-value (hanya bila urutan relevan):',round(acorr_breusch_godfrey(model,nlags=2)[1],4))
print('RESET p-value:',round(linear_reset(model,power=2,use_f=True).pvalue,4))
Xdesign=pd.DataFrame(model.model.exog,columns=model.model.exog_names)
vif=pd.DataFrame({'variabel':Xdesign.columns,'VIF':[variance_inflation_factor(Xdesign.values,i) for i in range(Xdesign.shape[1])]})
print(vif.round(2))
influence=OLSInfluence(model)
cook=np.asarray(influence.cooks_distance[0])
print('5 observasi dengan Cook distance terbesar:',np.argsort(cook)[-5:][::-1].tolist())
plt.figure(figsize=(9,3));plt.stem(np.arange(n),cook,markerfmt=',');plt.axhline(4/n,color='red',linestyle='--',label='Aturan praktis 4/n');plt.legend();plt.title('Cook distance');plt.show()

## 5. Evaluasi train/test (out-of-sample)
Data IID dipecah secara acak 80:20. Untuk data deret waktu, gunakan pemisahan kronologis. R² pada data uji dapat bernilai negatif.

In [ ]:
features=['jam_belajar','kehadiran','akses_internet']
X=df[features];y=df['nilai']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)
reg=LinearRegression().fit(X_train,y_train)
def metrics(name,actual,pred):
    return {'set':name,'MAE':mean_absolute_error(actual,pred),'RMSE':np.sqrt(mean_squared_error(actual,pred)),'R2':r2_score(actual,pred)}
results=pd.DataFrame([metrics('train',y_train,reg.predict(X_train)),metrics('test',y_test,reg.predict(X_test)),metrics('baseline mean',y_test,np.repeat(y_train.mean(),len(y_test)))])
print(results.round(3).to_string(index=False))
print('Intercept:',reg.intercept_,'Koefisien:',dict(zip(features,reg.coef_)))

## 6. Regularisasi Ridge dengan cross-validation
Standardisasi berada **di dalam pipeline** agar tidak bocor dari data validasi. RidgeCV memilih alpha menggunakan CV pada data latih saja.

In [ ]:
ridge=make_pipeline(StandardScaler(),RidgeCV(alphas=np.logspace(-3,3,30),cv=5))
ridge.fit(X_train,y_train)
print('Alpha terpilih:',ridge.named_steps['ridgecv'].alpha_)
print(pd.DataFrame([metrics('Ridge test',y_test,ridge.predict(X_test))]).round(3))

## 7. Simulasi pelanggaran asumsi: heteroskedastisitas dan nonlinieritas
Bandingkan residual plot, uji Breusch–Pagan, dan OLS standard error biasa vs HC3. Pada model salah spesifikasi, HC3 tidak otomatis menghilangkan bias.

In [ ]:
x=rng.uniform(0,10,350)
y_hetero=5+2*x+rng.normal(0,0.5+0.7*x,len(x))
y_curve=5+2*x+0.6*(x-5)**2+rng.normal(0,2,len(x))
for label,yy in [('Heteroskedastisitas',y_hetero),('Nonlinier',y_curve)]:
    temp=pd.DataFrame({'x':x,'y':yy})
    m=smf.ols('y ~ x',data=temp).fit()
    p_bp=het_breuschpagan(m.resid,m.model.exog)[1]
    fig,ax=plt.subplots(figsize=(6,3));ax.scatter(m.fittedvalues,m.resid,alpha=.5);ax.axhline(0,color='black',linestyle='--');ax.set(title=label+' — residual vs fitted',xlabel='Fitted',ylabel='Residual');plt.show()
    print(label,'BP p=',round(p_bp,5),'SE biasa=',round(m.bse['x'],3),'SE HC3=',round(m.get_robustcov_results(cov_type='HC3').bse[1],3))

## 8. Prediksi observasi baru dan interval
`get_prediction` menghasilkan interval kepercayaan rerata serta interval prediksi observasi individu. Interval prediksi umumnya lebih lebar.

In [ ]:
new=pd.DataFrame({'jam_belajar':[4,8,10],'kehadiran':[75,85,95],'akses_internet':[0,1,1]})
print(pd.concat([new,model.get_prediction(new).summary_frame(alpha=.05).reset_index(drop=True)],axis=1).round(2))

## 9. Ekspor data dan tugas mandiri
1. Ganti parameter noise menjadi heteroskedastik dan bandingkan hasil uji.
2. Tambahkan prediktor yang berkorelasi kuat dengan jam belajar dan periksa VIF.
3. Uji model dengan interaksi `jam_belajar:akses_internet`.
4. Gunakan dataset sendiri dan laporkan sumber, satuan, bias, CI, RMSE test, serta keterbatasan kausal.

**Peringatan:** dataset simulasi tidak boleh dilaporkan sebagai data empiris.

In [ ]:
df.to_csv('data_simulasi_pendidikan.csv',index=False)
print('Data disimpan sebagai data_simulasi_pendidikan.csv di sesi Colab.')
# Opsional: unduh dari Colab dengan:
# from google.colab import files
# files.download('data_simulasi_pendidikan.csv')